# create manual pairs

In [ ]:
import pandas as pd
#defendents_ethnicity
weapons = ["carrying_only", "importing_only", "possession_only"]
dfs = ["impute"]
for df_type in dfs:
    for weapon in weapons:
        df = pd.read_csv(
            rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\{df_type}\{weapon}.csv"
        )
            
      
        df_sorted = df.sort_values(
            by=["magazines", "bullets","handgun", "submachine_gun", "assault_rifle",
                "improvised_firearm", "grenade", "explosive_charge","actual_damage", "premeditation","escape_attempt", "undercover_sale", "has_independent_offenses",
                'year', 'Court', 'district', 'Location', 'judge_ethnic',
                'marital_status', 'age', 'Date'],
            ascending=[True] * 21
        ).reset_index(drop=True)

        # df_sorted = df.sort_values(
        #     by=["actual_damage", "premeditation","escape_attempt", "undercover_sale", "has_independent_offenses",
        #         "magazines", "bullets","handgun", "submachine_gun", "assault_rifle",
        #         "improvised_firearm", "grenade", "explosive_charge",
        #         'year', 'Court', 'district', 'Location', 'judge_ethnic',
        #         'marital_status', 'age', 'Date'],
        #     ascending=[True] * 21
        # ).reset_index(drop=True)

        df = df_sorted.reset_index(drop=True)
        pairs = []

        TREATMENT_VALUE = "Arab"
        CONTROL_VALUE = "Jewish"

        while len(df) > 1:
            treatment_rows = df[df["ethnic"] == TREATMENT_VALUE]
            if len(treatment_rows) == 0:
                break
            first_row = treatment_rows.iloc[0]
            candidates = df[df["ethnic"] == CONTROL_VALUE]
            if len(candidates) == 0:
                break
            distances = abs(candidates.index - first_row.name)
            closest_idx = candidates.index[distances.argmin()]
            matched_row = df.loc[closest_idx]
            pairs.append({
                "id_treat": first_row["original_case"],
                "id_control": matched_row["original_case"]
            })
            df = df.drop([first_row.name, closest_idx]).reset_index(drop=True)

        pairs_df = pd.DataFrame(pairs)
        pairs_df.to_csv(f"data/pairs/Defendants_Ethnic/{df_type}/{weapon}.csv", index=False)
        print(f"Created {len(pairs_df)} Arab-Jewish pairs for {weapon} ({df_type})")


In [ ]:
import pandas as pd
# age
weapons = ["carrying_only", "importing_only", "possession_only"]
dfs = ["impute"]
for df_type in dfs:
    for weapon in weapons:
        df = pd.read_csv(
            rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\{df_type}\{weapon}.csv"
        )

        df = df.sort_values(by=['magazines', 'bullets', 'weapon_motive', 'weapon_use',
                'year', 'Court', 'district', 'Location', 'judge_ethnic', 'ethnic',
                'marital_status', 'Date'],
            ascending=[True] * 12
        ).reset_index(drop=True)

        df = df.reset_index(drop=True)
        pairs = []
        TREATMENT_VALUE = "Young"
        CONTROL_VALUE = "Older"
        df["age_binary"] = df["age"].apply(lambda x: TREATMENT_VALUE if x <= 28 else CONTROL_VALUE)

        while len(df) > 1:
            treatment_rows = df[df["age_binary"] == TREATMENT_VALUE]
            if len(treatment_rows) == 0:
                break
            first_row = treatment_rows.iloc[0]
            candidates = df[df["age_binary"] == CONTROL_VALUE]
            if len(candidates) == 0:
                break
            distances = abs(candidates.index - first_row.name)
            closest_idx = candidates.index[distances.argmin()]
            matched_row = df.loc[closest_idx]
            pairs.append({
                "id_treat": first_row["original_case"],
                "id_control": matched_row["original_case"]
            })
            df = df.drop([first_row.name, closest_idx]).reset_index(drop=True)

        pairs_df = pd.DataFrame(pairs)
        pairs_df.to_csv(f"data/pairs/age/{df_type}/{weapon}.csv", index=False)
        print(f"Created {len(pairs_df)} age-based pairs for {weapon} ({df_type})")


In [ ]:
import pandas as pd
# year
weapons = ["carrying_only", "importing_only", "possession_only"]
dfs = ["impute"]
options = ['corona_period', 'war_period', 'pre_corona_vs_post_war', 'pre_corona_vs_pre_war', 'pre_war_vs_post_war']
for df_type in dfs:
    for weapon in weapons:
        for option in options:
            df = pd.read_csv(
                rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\{df_type}\{weapon}.csv"
            )

            df = df.sort_values(by=['magazines', 'bullets', 'weapon_motive', 'weapon_use',
                    'Court', 'district', 'Location', 'judge_ethnic', 'ethnic',
                    'marital_status', 'age', 'Date'],
                ascending=[True] * 12
            ).reset_index(drop=True)

            df = df.reset_index(drop=True)
            pairs = []
            TREATMENT_VALUE = 0
            CONTROL_VALUE = 1

            while len(df) > 1:
                treatment_rows = df[df[option] == TREATMENT_VALUE]
                if len(treatment_rows) == 0:
                    break
                first_row = treatment_rows.iloc[0]
                candidates = df[df[option] == CONTROL_VALUE]
                if len(candidates) == 0:
                    break
                distances = abs(candidates.index - first_row.name)
                closest_idx = candidates.index[distances.argmin()]
                matched_row = df.loc[closest_idx]
                pairs.append({
                    "id_treat": first_row["original_case"],
                    "id_control": matched_row["original_case"]
                })
                df = df.drop([first_row.name, closest_idx]).reset_index(drop=True)

            pairs_df = pd.DataFrame(pairs)
            pairs_df.to_csv(f"data/pairs/year/{df_type}/{weapon}_{option}.csv", index=False)
            print(f"Created {len(pairs_df)} year-based pairs for {weapon} ({df_type}) using {option}")


# build data

## fix age

In [ ]:
import pandas as pd
import re
from datetime import datetime
import math

# -------------------------
# Helpers
# -------------------------
def parse_iso(d):
    if d is None:
        return None

    # אם זה כבר date/datetime
    if isinstance(d, datetime):
        return d.date()

    s = str(d).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None

    # מחרוזות "רשימה ריקה" וכד'
    if s in {"[]", "[ ]", "{}", "{ }"}:
        return None

    # חלץ תאריך ISO מתוך טקסט (גם אם יש שעה)
    m = re.search(r"\b(\d{4}-\d{2}-\d{2})\b", s)
    if not m:
        return None

    try:
        return datetime.strptime(m.group(1), "%Y-%m-%d").date()
    except ValueError:
        return None


def norm_null(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() in {"nan", "none", "null"}:
            return None
        if s in {"[]", "[ ]", "{}", "{ }"}:
            return None
    return x


def compute_age_at_offense(extracted: dict):
    explicit = norm_null(extracted.get("explicit_age_at_offense"))
    if explicit is not None:
        # explicit should be numeric; if it's a string, extract digits
        if isinstance(explicit, str):
            m = re.search(r"\b(\d{1,3})\b", explicit)
            explicit = int(m.group(1)) if m else None
        if explicit is not None:
            return int(explicit)

    offense_date = norm_null(extracted.get("offense_date"))
    offense_dates = extracted.get("offense_dates") or []
    sentencing_date = norm_null(extracted.get("sentencing_date"))

    if (offense_date is None) and offense_dates:
        if sentencing_date:
            sd = parse_iso(sentencing_date)
            ods = [parse_iso(x) for x in offense_dates if x]
            ods = [x for x in ods if x]
            if ods:
                offense_date = min(ods, key=lambda x: abs((sd - x).days))
        if offense_date is None:
            offense_date = offense_dates[0]

    od = parse_iso(offense_date) if offense_date else None

    bd = parse_iso(norm_null(extracted.get("birth_date")))
    if bd and od:
        return math.floor((od - bd).days / 365.25)

    age_now = norm_null(extracted.get("age_now"))
    if isinstance(age_now, str):
        m = re.search(r"\b(\d{1,3})\b", age_now)
        age_now = int(m.group(1)) if m else None

    ref_date_str = norm_null(extracted.get("age_reference_date")) or norm_null(extracted.get("sentencing_date"))
    ref = parse_iso(ref_date_str) if ref_date_str else None

    if age_now is not None and ref and od:
        delta_years = ((ref - od).days / 365.25)
        #round down or up to nearest integer
        
        delta_years = math.floor(((ref - od).days / 365.25))
        return int(age_now) - delta_years
    
    # --- fallback: birth_year + offense_date ---
    birth_year = norm_null(extracted.get("birth_year"))
    if birth_year is not None and od:
        try:
            by = int(birth_year)
            return od.year - by
        except Exception:
            pass

    # --- fallback: birth_year + sentencing_date ---
    if birth_year is not None and sentencing_date:
        sd = parse_iso(sentencing_date)
        if sd:
            try:
                by = int(birth_year)
                return sd.year - by
            except Exception:
                pass
    if age_now is not None:
        return int(age_now)
    
    age_mentions = extracted.get("age_mentions") or []

    if age_mentions:
        try:
            return int(min(age_mentions))  # או max / min לפי מה שמתאים לך
        except:
            pass
    return "לא ידוע"


OFFENSE_TRIGGERS = ["בעת ביצוע", "ביום האירוע", "במועד ביצוע", "בזמן ביצוע", "במועד האירוע", "במועד הרלוונטי"]

def sanitize_explicit_age(d):
    """
    If explicit_age_at_offense is a string like 'הנאשם בן 43',
    keep it ONLY if the same string contains an offense-time trigger.
    Otherwise set to None (it's age_now, not explicit at offense).
    """
    x = d.get("explicit_age_at_offense")
    if isinstance(x, str):
        m = re.search(r"\b(\d{1,3})\b", x)
        num = int(m.group(1)) if m else None

        if any(t in x for t in OFFENSE_TRIGGERS):
            d["explicit_age_at_offense"] = num
        else:
            d["explicit_age_at_offense"] = None
            # optional: if age_now missing, use the number as age_now
            if d.get("age_now") in (None, "", "nan") and num is not None:
                d["age_now"] = num
    return d

def parse_offense_dates_cell(val):
    """
    offense_dates might be:
    - NaN / empty
    - a single ISO string
    - a stringified list like "['2021-06-05', '2021-06-10']"
    - a JSON list string like '["2021-06-05"]'
    Normalize to list[str].
    """
    if val is None:
        return []
    s = str(val).strip()
    if s == "" or s.lower() == "nan":
        return []

    # already looks like a list
    if s.startswith("[") and s.endswith("]"):
        # try JSON style first
        try:
            import json
            out = json.loads(s)
            if isinstance(out, list):
                return [str(x).strip().strip('"').strip("'") for x in out if str(x).strip()]
        except Exception:
            pass
        # fallback: extract ISO dates by regex
        return re.findall(r"\d{4}-\d{2}-\d{2}", s)

    # single date
    m = re.search(r"\d{4}-\d{2}-\d{2}", s)
    return [m.group(0)] if m else []

# -------------------------
# Load
# -------------------------
df = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv",
    encoding="utf-8-sig"
)

# -------------------------
# Compute new column
# -------------------------
AGE_FIELDS = [
    "explicit_age_at_offense",
    "age_now",
    "age_reference_date",
    "offense_date",
    "offense_dates",
    "sentencing_date",
    "birth_date",
    "birth_year",
]

def compute_age_from_columns(row):
    extracted = {k: norm_null(row.get(k, None)) for k in AGE_FIELDS}

    extracted["offense_dates"] = parse_offense_dates_cell(extracted.get("offense_dates"))
    extracted = sanitize_explicit_age(extracted)

    return compute_age_at_offense(extracted)


# Create a NEW column (don’t overwrite your extracted columns)
df["age"] = df.apply(compute_age_from_columns, axis=1)

# -------------------------
out_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved:", out_path)


## add punishment months

In [ ]:
import pandas as pd

PRIORITY_ORDER = [
    "בפועל",
    "עבודות שירות",
    "על תנאי",
    "שלצ",
    "קנס"
]
import re
def extract_months(text: str) -> float:
    """
    Extract numeric value from strings like:
    '66 חודשי מאסר', '1.17 חודשי מאסר בעבודות שירות'
    """
    if not isinstance(text, str) or not text.strip():
        return 0.0
    match = re.search(r'(\d+(\.\d+)?)', text)
    return float(match.group(1)) if match else 0.0

def build_sentence_columns(df: pd.DataFrame) -> pd.DataFrame:
    sentence_types = []
    sentence_months = []
    punishment_months = []
    is_actual_sentences = []

    for _, row in df.iterrows():
        chosen_type = ""
        chosen_value = ""

        # choose sentence type by priority
        for col in PRIORITY_ORDER:
            if col in df.columns and isinstance(row[col], str) and row[col].strip():
                chosen_type = col
                chosen_value = row[col]
                break

        months = extract_months(chosen_value)

        # is_actual_sentence
        is_actual = chosen_type in ["בפועל", "עבודות שירות"]

        # punishment_months (custodial)
        pm = months if is_actual else 0.0

        sentence_types.append(chosen_type)
        sentence_months.append(months)
        punishment_months.append(pm)
        is_actual_sentences.append(is_actual)


    df["sentence_type"] = sentence_types
    df["sentence_months"] = sentence_months
    df["punishment_months"] = punishment_months
    df['is_actual_sentence'] = is_actual_sentences

    return df
df = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv",
    encoding="utf-8-sig"
)
df = build_sentence_columns(df)
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", index=False, encoding="utf-8-sig")


## keep relevance columns

In [ ]:
import pandas as pd
df = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv",
    encoding="utf-8-sig"
)

columns_to_keep = ['original_case',
       'section_144a', 'section_144b', 'section_144b2',
       'magazines', 'bullets', 'confession',
       'Criminal_record', 'actual_damage', 'potential_damage',
       'escape_attempt', 'has_independent_offenses', 'marital_status', 'premeditation',
       'regret', 'undercover_sale', 'weapon_motive',
       'weapon_storage', 'handgun',
       'submachine_gun', 'assault_rifle', 'sniper_rifle', 'hunting_rifle',
       'improvised_firearm', 'grenade', 'explosive_charge', 'molotov',
       'anti_tank_missile', 'ammunition_only', 'weapon_parts',
       'unidentified_firearm', 'weapon_use', 'age', 'punishment_months',
       'is_actual_sentence']

df = df[columns_to_keep]
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd
df = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv",
    encoding="utf-8-sig"
)

print(df.columns)



## orginaize columns

In [ ]:
import pandas as pd
file_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")
df['Criminal_record'] = df['Criminal_record'].str.replace('"', '')
# if value is in [לא עבר פלילי,לאה עבר פלילי, לא ללא עבר פלילי] change to ללא עבר פלילי
df['Criminal_record'] = df['Criminal_record'].apply(lambda x: 'ללא עבר פלילי' if x in ['לא עבר פלילי', 'לאה עבר פלילי', 'לא ללא עבר פלילי'] else x)
df['has_independent_offenses'] = df['has_independent_offenses'].str.replace('"', '')
df['regret'] = df['regret'].str.replace('"', '')
df['weapon_use'] = df['weapon_use'].apply(lambda x: 'החזקה בלבד' if x in ['החזוקה בלבד', 'החזקת בלבד', 'החזקה בלבד - הנשק לא שומש בפועל'] else x)

df.to_csv(file_path, index=False, encoding="utf-8-sig")

## remove more than 1 defendants

In [ ]:
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", encoding="utf-8-sig")
print("Initial rows:", len(df))

df['accused_count'] = df['בפועל'].apply(lambda x: str(x).count('נאשם'))
df = df[df['accused_count'] <= 1] 

print("Rows after removing multiple defendants:", len(df))
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", index=False, encoding="utf-8-sig")

## parse the punishment and add metadata

In [ ]:
import pandas as pd
import re
from typing import Dict, Any
import pandas as pd
from datetime import datetime

import pandas as pd
import re


# Helper function to convert section column to binary (1/0)
def to_binary(val):
    if pd.isna(val):
        return 0
    val_str = str(val).strip().lower()
    if val_str in ['', 'לא', 'nan']:
        return 0
    return 1


location_mapping = {
    "תל אביב-יפו": "Tel Aviv",

    "ראשון לציון": "Center",
    "רמלה": "Center",
    "רחובות": "Center",
    "פתח תקווה": "Center",
    "פתח תקוה": "Center",
    "מרכז": "Center",
    "כפר סבא": "Center",
    "נתניה": "Center",

    "באר שבע": "South",
    "קריית גת": "South",
    "ק' גת": "South",
    "אשקלון": "South",
    "אילת": "South",
    "אשדוד": "South",
    "דימונה": "South",

    "בית שמש": "Jerusalem",
    "ירושלים": "Jerusalem",

    "קי שמונה": "North",  # move to "North" if you meant קריית שמונה
    "ק' שמונה": "North",
    "נצרת": "North",
    "טבריה": "North",
    "עפולה": "North",
    "קצרין": "North",
    "צפת": "North",
    "בית שאן": "North",

    "חיפה": "Haifa",
    "חדרה": "Haifa",
    "עכו": "Haifa",
    "קריות": "Haifa"
}

# Load data
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", encoding="utf-8-sig")
print("Initial rows:", len(df))
# Apply parser
df = build_sentence_columns(df)

# Extract just the number from the drug column
# for col in drug_cols:
#     df[col] = df[col].apply(extract_quantity)

# Create new offense columns
df['possession_offence'] = df['section_144a'].apply(to_binary)       
df['carrying_offence'] = df['section_144b'].apply(to_binary)              
df['importing_offence'] = df['section_144b2'].apply(to_binary)    

# Columns to add from verdicts
cols_to_add = ['Date', 'Court', 'Location', 'Judges', 'Prosecutors', 'Defendants']

# Load the verdicts CSV with additional columns
verdicts_df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\defendants\one_defendants.csv", encoding="utf-8-sig")

# Merge with verdicts to add Date, Court, Location, Judges, Prosecutors, Defendants
df = df.merge(
    verdicts_df[['file_name'] + cols_to_add],
    left_on='original_case',
    right_on='file_name',
    how='left'
)

# Drop duplicate verdict column if exists
if 'verfile_namedict' in df.columns and 'original_case' in df.columns:
    df = df.drop(columns=['file_name'])
print(df.columns)
# add year column extracted from Date
# If format is DD/MM/YYYY (Israeli format)
df['year'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce').dt.year

# If format is DD-MM-YYYY
df['year'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce').dt.year

# If format is YYYY-MM-DD
df['year'] = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce').dt.year

# If mixed formats, try dayfirst=True
df['year'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce').dt.year

# Map Location to region
df['district'] = df['Location'].map(location_mapping).fillna('Other')

all_gender = []
verdicts =[]
for row in df['original_case']:
    # lst = ['SH-14-09-23152-186', 'ME-13-12-33828-11', 'SH-15-04-21459-877', 'SH-16-08-39727-191', 'ME-12-08-3155-55']
    # if row in lst:
    #     all_gender.append("unknown")
    path = rf'C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\CSV_OUTPUT\{row}.csv'
    df_case = pd.read_csv(path, encoding="utf-8-sig")
    for i in df_case['text']:
        if 'נאשמת' in i:
            all_gender.append('female')
            verdicts.append(row)
            break
        elif 'נאשם' in i:
            verdicts.append(row)
            all_gender.append('male')
            break
#print verdict differences
print(set(df['original_case'].tolist()) - set(verdicts))
# add gender column to df
df['gender'] = all_gender 

CORONA = pd.Timestamp("2020-03-11")
WAR = pd.Timestamp("2023-10-07")
df["date_dt"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

df["corona_period"] = df["date_dt"].apply(
    lambda x: 0 if x < CORONA else 1
)
# Create a column for cases that are post-corona but pre-war


df["war_period"] = df["date_dt"].apply(
    lambda x: 0 if x < WAR else 1
)

# pre corona vs post war
def classify_corona_war(date):
    if pd.isna(date):
        return None
    elif date < CORONA:
        return 0
    elif date >= WAR:
        return 1
    else:
        return None  # "between" period

df["pre_corona_vs_post_war"] = df["date_dt"].apply(classify_corona_war)

def classify_corona(date):
    if pd.isna(date):
        return None
    elif date < CORONA:
        return 0
    elif date > CORONA and date < WAR:
        return 1
    else:
        return None
    
df["pre_corona_vs_pre_war"] = df["date_dt"].apply(classify_corona)

def classify_war(date):
    if pd.isna(date):
        return None
    elif date < WAR and date > CORONA:
        return 0
    elif date >= WAR:
        return 1
    else:
        return None
df["pre_war_vs_post_war"] = df["date_dt"].apply(classify_war)

# #remove mdmb column if exists
# if 'mdmb' in df.columns:
#     df = df.drop(columns=['mdmb'])
print("Final rows:", len(df))

# if row (in file_name) is not in df, remove the row from df
df = df[df['original_case'].isin(verdicts)]
print(len(df))


# # Save
# df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", index=False, encoding="utf-8-sig")


In [ ]:
verdicts_df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\defendants\one_defendants.csv", encoding="utf-8-sig")
# remove rows from df if original_case is not in verdicts_df['file_name']
df = df[df['original_case'].isin(verdicts_df['file_name'])]
print("Rows after syncing with verdicts_df:", len(df))
# Save
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", index=False, encoding="utf-8-sig")

In [ ]:
#column to remove
columns_to_remove = ['section_144a', 'section_144b', 'section_144b2','date_dt','sentence_months', 'sentence_type','file_name']

df = df.drop(columns=columns_to_remove)
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", index=False, encoding="utf-8-sig")

## add punishment range

In [ ]:
import pandas as pd

LOWER_BOUND_DEFAULT = 3  # default minimum punishment in months when only max is known

df = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\crawler\batch_results_CSV_OUTPUT.csv",
    encoding="utf-8-sig"
)
df1 = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv",
    encoding="utf-8-sig"
)


# Merge
df1 = df1.merge(
    df[['verdict', 'range_low', 'range_high']],
    left_on='original_case',
    right_on='verdict',
    how='left'
)

# Rename to your schema
df1 = df1.rename(columns={
    "range_low": "min_punishment",
    "range_high": "max_punishment"
})
def fill_min_punishment(row):
    if pd.isna(row['min_punishment']) and pd.notna(row['max_punishment']):
        return LOWER_BOUND_DEFAULT
    return row['min_punishment']
df1['min_punishment'] = df1.apply(fill_min_punishment, axis=1)


df1['relative_range_width'] = (
    (df1['max_punishment'] - df1['min_punishment']) /
    df1['max_punishment']
).where(df1['max_punishment'] > 0)

# remove 'verdict' column if exists
if 'verdict' in df1.columns:
    df1 = df1.drop(columns=['verdict'])
    
df1.to_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv",
    index=False,
    encoding="utf-8-sig"
)


## replace to 1/0

In [ ]:
binary_cols = ['confession', 'regret','escape_attempt','has_independent_offenses','premeditation','undercover_sale']  # הוסף עוד אם צריך
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", encoding="utf-8-sig")
# df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\drug_features_with_punishmentsRanges_cleanedAge.csv", encoding="utf-8-sig")

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].replace({'כן': 1, 'לא': 0})
        df[col] = pd.to_numeric(df[col], errors='coerce')


# שמור
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", index=False, encoding="utf-8-sig")


## remove female

In [ ]:
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", encoding="utf-8-sig")
# remove rows where in gender column the value is female 
print(len(df))
df = df[df['gender'] != 'female']
print(len(df))
# save the result to the same file
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", index=False, encoding="utf-8-sig")

## remove rows with no punishment info

In [ ]:
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", encoding="utf-8-sig")

#remove rows where both range_low or range_high are na
print("Initial rows:", len(df))
# print the rows where  min_punishment and  max_punishment is na
print(
    "Rows with missing min_punishment and max_punishment:\n",
    len(df[df["max_punishment"].isna() & df["min_punishment"].isna()])
)
print("Rows with missing min_punishment:\n", len(df[df['min_punishment'].isna()]))
print("Rows with missing max_punishment:\n", len(df[df['max_punishment'].isna()]))
df = df[~((df['max_punishment'].isna()) & (df['min_punishment'].isna()) | df['punishment_months'].isna())]
print("Rows after removing missing punishments:", len(df))
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", index=False, encoding="utf-8-sig")
print("✅ Saved!")


## split to possesion, importation, trading

In [ ]:
import pandas as pd
import re

# Load the normalized drug features
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", encoding="utf-8-sig")

# for possession_offence, carrying_offence,importing_offence split the data of each time every offence appears alone
possession_only = df[(df['possession_offence'] == 1) & (df['carrying_offence'] == 0) & (df['importing_offence'] == 0)]
carrying_only   = df[(df['carrying_offence'] == 1)   & (df['possession_offence'] == 0) & (df['importing_offence'] == 0)]
importing_only  = df[(df['importing_offence'] == 1)  & (df['possession_offence'] == 0) & (df['carrying_offence'] == 0)]

print(f"possession only: {len(possession_only)}")
print(f"carrying only:   {len(carrying_only)}")
print(f"importing only:  {len(importing_only)}")

#print ethnic value counts in each subset
print("Ethnic counts in possession only:\n", possession_only['ethnic'].value_counts())
print("Ethnic counts in carrying only:\n", carrying_only['ethnic'].value_counts())
print("Ethnic counts in importing only:\n", importing_only['ethnic'].value_counts())

# remove columns
# columns_to_remove = ['carrying_offence', 'importing_offence', 'possession_offence']
# possession_only = possession_only.drop(columns=columns_to_remove)
# carrying_only = carrying_only.drop(columns=columns_to_remove)
# importing_only = importing_only.drop(columns=columns_to_remove) 
# # Save each subset to a new CSV
# possession_only.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute\possession_only.csv", index=False, encoding="utf-8-sig")
# carrying_only.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute\carrying_only.csv", index=False, encoding="utf-8-sig")
# importing_only.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute\importing_only.csv", index=False, encoding="utf-8-sig")



## imputed missing ages and status

In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# Ordinal encoding for Criminal_record (ordered: no record < light < heavy)
CRIMINAL_RECORD_ORDER = {
    'ללא עבר פלילי': 0,
    'עבר פלילי קל':  1,
    'עבר פלילי כבד': 2,
}
CRIMINAL_RECORD_DECODE = {v: k for k, v in CRIMINAL_RECORD_ORDER.items()}

offence = ['carrying_only', 'importing_only', 'possession_only']
for off in offence:
    df = pd.read_csv(
        rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute\{off}.csv",
        encoding="utf-8-sig"
    )

    # --- נקה age ---
    def clean_age(val):
        if pd.isna(val):
            return np.nan
        val_str = str(val).strip()
        if val_str == 'לא ידוע' or val_str == '':
            return np.nan
        try:
            return float(val_str)
        except:
            return np.nan

    df['age'] = df['age'].apply(clean_age)

    # --- נקה marital_status והמר לקודים ---
    df['marital_status'] = df['marital_status'].replace('לא ידוע', np.nan)

    status_categories = df['marital_status'].dropna().unique().tolist()
    status_to_code = {status: i for i, status in enumerate(status_categories)}
    code_to_status = {i: status for status, i in status_to_code.items()}
    df['marital_status_code'] = df['marital_status'].map(status_to_code)

    print(f"Marital status mapping: {status_to_code}")

    # --- נקה Criminal_record והמר לקודים אורדינליים ---
    df['Criminal_record'] = df['Criminal_record'].replace('לא ידוע', np.nan)
    df['criminal_record_code'] = df['Criminal_record'].map(CRIMINAL_RECORD_ORDER)

    print(f"Criminal record missing before imputation: {df['Criminal_record'].isna().sum()}")

    # --- הכן משתנים ---
    cols_to_impute = ['age', 'marital_status_code', 'criminal_record_code']
    helper_numeric = [
        'confession', 'regret', 'premeditation', 'undercover_sale',
        'escape_attempt', 'has_independent_offenses',
        'magazines', 'bullets',
        'handgun', 'submachine_gun', 'assault_rifle', 'sniper_rifle',
        'hunting_rifle', 'improvised_firearm', 'grenade', 'explosive_charge',
        'molotov', 'anti_tank_missile', 'ammunition_only', 'weapon_parts',
        'unidentified_firearm',
        'year'
    ]
    helper_numeric = [c for c in helper_numeric if c in df.columns]
    for col in helper_numeric:
        if col in df.columns:
            df[col] = df[col].replace({'כן': 1, 'לא': 0})
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # עמודות עזר (קטגוריאליות)
    helper_categorical = ['district', 'Court', 'ethnic', 'judge_ethnic']
    helper_categorical = [c for c in helper_categorical if c in df.columns]

    # One-hot encoding לקטגוריאליות
    df_encoded = pd.get_dummies(df[helper_categorical], dummy_na=False)

    # חבר הכל
    df_for_imputation = pd.concat([
        df[cols_to_impute],
        df[helper_numeric],
        df_encoded
    ], axis=1)

    # שמור מקוריים (חוץ מהעמודות שרוצים לחזות)
    original_values = {col: df_for_imputation[col].copy()
                    for col in df_for_imputation.columns
                    if col not in cols_to_impute}

    # --- MICE עם Random Forest ---
    imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=100, random_state=42),
        max_iter=10,
        random_state=42
    )
    df_imputed = pd.DataFrame(
        imputer.fit_transform(df_for_imputation),
        columns=df_for_imputation.columns
    )

    # --- החזר ערכים ---
    # age
    df['age'] = df_imputed['age'].round().astype(int)

    # marital_status - עגל והמר בחזרה
    df['marital_status_code'] = df_imputed['marital_status_code'].round().astype(int)
    df['marital_status_code'] = df['marital_status_code'].clip(0, len(status_categories) - 1)
    df['marital_status'] = df['marital_status_code'].map(code_to_status)
    df = df.drop(columns=['marital_status_code'])

    # Criminal_record - עגל והמר בחזרה
    criminal_codes = df_imputed['criminal_record_code'].round().astype(int).clip(0, 2)
    df['Criminal_record'] = criminal_codes.map(CRIMINAL_RECORD_DECODE)
    df = df.drop(columns=['criminal_record_code'], errors='ignore')

    # החזר ערכים מקוריים לעמודות העזר
    for col in original_values:
        if col in df.columns:
            df[col] = original_values[col]

    # --- בדיקה ---
    print(f"\n✅ {off} after imputation:")
    print(f"  Age missing:             {df['age'].isna().sum()}")
    print(f"  Marital status missing:  {df['marital_status'].isna().sum()}")
    print(f"  Criminal_record missing: {df['Criminal_record'].isna().sum()}")
    print(f"  Criminal_record dist:\n{df['Criminal_record'].value_counts().to_string()}")

    # שמור
    df.to_csv(
        rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute\{off}.csv",
        index=False, encoding="utf-8-sig"
    )


# change  in punishment range into 0/6

In [ ]:
possession_only = pd.read_csv(
rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute\possession_only.csv",
encoding="utf-8-sig"
)

carrying_only = pd.read_csv(
rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute\carrying_only.csv",
encoding="utf-8-sig"
)  

importing_only = pd.read_csv(
rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute\importing_only.csv",
encoding="utf-8-sig"
)  
for lower in LOWER_BOUND_DEFAULT:
    possession_only.to_csv(rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute_{lower}\possession_only.csv", index=False, encoding="utf-8-sig")
    carrying_only.to_csv(rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute_{lower}\carrying_only.csv", index=False, encoding="utf-8-sig")
    importing_only.to_csv(rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute_{lower}\importing_only.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd

LOWER_BOUND_DEFAULT = [0,6]

df = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\crawler\batch_results_CSV_OUTPUT.csv",
    encoding="utf-8-sig"
)

df2 = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv",
    encoding="utf-8-sig"
)

dfs = ['possession_only', 'carrying_only','importing_only']
for lower in LOWER_BOUND_DEFAULT:
    for f in dfs:
        df1 = pd.read_csv(
        rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute_{lower}\{f}.csv",
        encoding="utf-8-sig"
        )


        # Merge
        df1 = df1.merge(
            df[['verdict', 'range_low', 'range_high']],
            left_on='original_case',
            right_on='verdict',
            how='left'
        )

        # find rows in df1 where pd.isna(row['min_punishment']) and pd.notna(row['max_punishment']) and verdict is in df2['verdict']
        missing_min = df1[(df1['range_low'].isna()) & (df1['range_high'].notna()) & (df1['min_punishment'] == 3)]
        print(len(missing_min))
    # FOR missing_min CHANGE IN df2 THE MIN INTO LOWER_BOUND_DEFAULT AND SAVE CSV
        df1.loc[df1['original_case'].isin(missing_min['original_case']), 'min_punishment'] = lower
        df1.to_csv(
            rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\impute_{lower}\{f}.csv",
            index=False, encoding="utf-8-sig"
        )

# SM sensitivity data
Data variants for the robustness checks in Section 3.3. These cells only write to `data/no_impute`; `data/impute` is not modified.

## no_impute – listwise deletion (Section 3.3.2a)
Start from the primary `impute` files, restore the three imputed columns to their original values, and drop every case where any of them is unknown. All other columns stay identical to the primary analysis.

In [ ]:
import pandas as pd

BASE = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data"
UNKNOWN = "לא ידוע"
PERSONAL_COLS = ["age", "marital_status", "Criminal_record"]
OFFENCES = ["possession_only", "carrying_only", "importing_only"]

# Original (pre-imputation) personal values, by case
features = pd.read_csv(rf"{BASE}\features_extractions.csv", encoding="utf-8-sig")
original_personal = features.set_index("original_case")[PERSONAL_COLS]

def revert_imputation(df):
    df = df.copy()
    df[PERSONAL_COLS] = original_personal.loc[df["original_case"]].values
    return df

for off in OFFENCES:
    df = revert_imputation(pd.read_csv(rf"{BASE}\impute\{off}.csv", encoding="utf-8-sig"))
    known = (df[PERSONAL_COLS].astype(str) != UNKNOWN).all(axis=1)
    df[known].to_csv(rf"{BASE}\no_impute\{off}.csv", index=False, encoding="utf-8-sig")
    print(f"✅ {off}: kept {known.sum()} of {len(df)} rows")
    print(df[known]["ethnic"].value_counts().to_string(), "\n")

## unfiltered – cases without a declared sentencing range (Section 3.3.3)
The 1,302-case stage (single male defendant) before excluding verdicts that lack a sentencing range. The 1,148 cases in the final sample are taken from the primary `impute` files (imputed columns restored); the 154 excluded cases are built from the raw extraction (`features_extractions1.csv`) and `initial_data/one_defendants.csv` with the same transformations. Age uses the rule of the primary data: stated age at offense, otherwise current age. Imputation is the same as the primary `impute` data. Only `is_actual_sentence` is analysed on this sample.

In [ ]:
# Uses BASE, UNKNOWN, features and revert_imputation from the no_impute cell above
import re
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

raw = pd.read_csv(rf"{BASE}\features_extractions1.csv", encoding="utf-8-sig")
one = pd.read_csv(rf"{BASE}\initial_data\one_defendants.csv", encoding="utf-8-sig")
CSV_OUTPUT = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\CSV_OUTPUT"

# --- 1. Excluded cases: single defendant, male, not in the final sample ---
def defendant_gender(case):
    for text in pd.read_csv(rf"{CSV_OUTPUT}\{case}.csv", encoding="utf-8-sig")["text"]:
        if "נאשמת" in str(text):
            return "female"
        if "נאשם" in str(text):
            return "male"
    return None

excluded = raw[raw["original_case"].isin(one["file_name"]) & ~raw["original_case"].isin(features["original_case"])].copy()
excluded["gender"] = excluded["original_case"].apply(defendant_gender)
excluded = excluded[excluded["gender"] != "female"]
print(f"Excluded cases (no sentencing range): {len(excluded)}")

# --- 2. Same transformations as the primary data ---
def to_binary(val):
    if pd.isna(val):
        return 0
    return 0 if str(val).strip().lower() in ['', 'לא', 'nan'] else 1

def stated_age(row):
    for col in ["explicit_age_at_offense", "age_now"]:
        if pd.notna(row[col]):
            m = re.search(r"\d{1,3}", str(row[col]))
            if m:
                return m.group()
    return UNKNOWN

STORAGE_TOKENS = {"בבית": "בית", "ברכב": "רכב", "במקום ציבורי": "מקום ציבורי"}
def normalize_storage(val):
    if pd.isna(val):
        return val
    tokens = {STORAGE_TOKENS.get(t.strip(), t.strip()) for t in str(val).split(",") if t.strip()}
    return ",".join(sorted(tokens))

binary_cols = ['confession', 'regret', 'escape_attempt', 'has_independent_offenses', 'premeditation', 'undercover_sale']
for col in binary_cols + ['Criminal_record']:
    excluded[col] = excluded[col].astype(str).str.replace('"', '').replace('nan', np.nan)
for col in binary_cols:
    excluded[col] = pd.to_numeric(excluded[col].replace({'כן': 1, 'לא': 0}), errors='coerce')
excluded['Criminal_record'] = excluded['Criminal_record'].apply(
    lambda x: 'ללא עבר פלילי' if x in ['לא עבר פלילי', 'לאה עבר פלילי', 'לא ללא עבר פלילי'] else x)
excluded['weapon_use'] = excluded['weapon_use'].apply(
    lambda x: 'החזקה בלבד' if x in ['החזוקה בלבד', 'החזקת בלבד', 'החזקה בלבד - הנשק לא שומש בפועל'] else x)
excluded['weapon_storage'] = excluded['weapon_storage'].apply(normalize_storage)
excluded['age'] = excluded.apply(stated_age, axis=1)
excluded['possession_offence'] = excluded['section_144a'].apply(to_binary)
excluded['carrying_offence']   = excluded['section_144b'].apply(to_binary)
excluded['importing_offence']  = excluded['section_144b2'].apply(to_binary)

meta_cols = ['Date', 'Court', 'Location', 'Judges', 'Prosecutors', 'Defendants', 'district', 'ethnic', 'judge_ethnic']
excluded = excluded.merge(one[['file_name'] + meta_cols], left_on='original_case', right_on='file_name', how='left')
date_dt = pd.to_datetime(excluded['Date'], dayfirst=True, errors='coerce')
excluded['Date'] = date_dt.dt.strftime('%d/%m/%Y')
excluded['year'] = date_dt.dt.year
CORONA, WAR = pd.Timestamp("2020-03-11"), pd.Timestamp("2023-10-07")
excluded['corona_period'] = (date_dt >= CORONA).astype(int)
excluded['war_period'] = (date_dt >= WAR).astype(int)
excluded['pre_corona_vs_post_war'] = np.select([date_dt < CORONA, date_dt >= WAR], [0, 1], np.nan)
excluded['pre_corona_vs_pre_war'] = np.select([date_dt < CORONA, (date_dt > CORONA) & (date_dt < WAR)], [0, 1], np.nan)
excluded['pre_war_vs_post_war'] = np.select([(date_dt > CORONA) & (date_dt < WAR), date_dt >= WAR], [0, 1], np.nan)
print(excluded["ethnic"].value_counts().to_string(), "\n")

# --- 3. Imputation (same MICE + random forest as the primary `impute` data) ---
CRIMINAL_RECORD_ORDER = {
    'ללא עבר פלילי': 0,
    'עבר פלילי קל':  1,
    'עבר פלילי כבד': 2,
}
CRIMINAL_RECORD_DECODE = {v: k for k, v in CRIMINAL_RECORD_ORDER.items()}

def impute_personal(df):
    df = df.reset_index(drop=True)
    df['age'] = pd.to_numeric(df['age'].replace('לא ידוע', np.nan), errors='coerce')

    df['marital_status'] = df['marital_status'].replace('לא ידוע', np.nan)
    status_categories = df['marital_status'].dropna().unique().tolist()
    status_to_code = {status: i for i, status in enumerate(status_categories)}
    code_to_status = {i: status for status, i in status_to_code.items()}
    df['marital_status_code'] = df['marital_status'].map(status_to_code)

    df['Criminal_record'] = df['Criminal_record'].replace('לא ידוע', np.nan)
    df['criminal_record_code'] = df['Criminal_record'].map(CRIMINAL_RECORD_ORDER)

    cols_to_impute = ['age', 'marital_status_code', 'criminal_record_code']
    helper_numeric = [
        'confession', 'regret', 'premeditation', 'undercover_sale',
        'escape_attempt', 'has_independent_offenses',
        'magazines', 'bullets',
        'handgun', 'submachine_gun', 'assault_rifle', 'sniper_rifle',
        'hunting_rifle', 'improvised_firearm', 'grenade', 'explosive_charge',
        'molotov', 'anti_tank_missile', 'ammunition_only', 'weapon_parts',
        'unidentified_firearm',
        'year'
    ]
    helper_numeric = [c for c in helper_numeric if c in df.columns]
    for col in helper_numeric:
        df[col] = pd.to_numeric(df[col].replace({'כן': 1, 'לא': 0}), errors='coerce')
    helper_categorical = ['district', 'Court', 'ethnic', 'judge_ethnic']
    helper_categorical = [c for c in helper_categorical if c in df.columns]
    df_encoded = pd.get_dummies(df[helper_categorical], dummy_na=False)

    df_for_imputation = pd.concat([df[cols_to_impute], df[helper_numeric], df_encoded], axis=1)
    original_values = {col: df_for_imputation[col].copy()
                       for col in df_for_imputation.columns if col not in cols_to_impute}

    imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=100, random_state=42),
        max_iter=10,
        random_state=42
    )
    df_imputed = pd.DataFrame(imputer.fit_transform(df_for_imputation), columns=df_for_imputation.columns)

    df['age'] = df_imputed['age'].round().astype(int)
    df['marital_status_code'] = df_imputed['marital_status_code'].round().astype(int).clip(0, len(status_categories) - 1)
    df['marital_status'] = df['marital_status_code'].map(code_to_status)
    df = df.drop(columns=['marital_status_code'])
    df['Criminal_record'] = df_imputed['criminal_record_code'].round().astype(int).clip(0, 2).map(CRIMINAL_RECORD_DECODE)
    df = df.drop(columns=['criminal_record_code'], errors='ignore')
    for col in original_values:
        if col in df.columns:
            df[col] = original_values[col]
    return df

# --- 4. Split to single-offense types, add to the filtered cases, impute, save ---
OFFENCE_COLS = {"possession_only": "possession_offence",
                "carrying_only":   "carrying_offence",
                "importing_only":  "importing_offence"}
for off, col in OFFENCE_COLS.items():
    others = [c for c in OFFENCE_COLS.values() if c != col]
    kept = revert_imputation(pd.read_csv(rf"{BASE}\impute\{off}.csv", encoding="utf-8-sig"))
    added = excluded[(excluded[col] == 1) & (excluded[others[0]] == 0) & (excluded[others[1]] == 0)]
    added = added.reindex(columns=kept.columns)
    df = impute_personal(pd.concat([kept, added], ignore_index=True))
    df.to_csv(rf"{BASE}\unfiltered\{off}.csv", index=False, encoding="utf-8-sig")
    print(f"✅ {off}: {len(kept)} filtered + {len(added)} excluded = {len(df)} rows")
    print(df["ethnic"].value_counts().to_string(), "\n")